# Assignment 1, Group 3
__*Fernando Green, Tyler Hensel, Jacob Ramer, Elise Wallace*__


## Importing Packages and Data

In [136]:
import pandas as pd
import numpy as np

In [137]:
transaction_data = pd.read_csv("../data/transaction_data.csv")
meta_data = pd.read_csv("../data/meta_data.csv")

## Goal of the Exercise

It is important to keep the overarching goal of this exercise, which is to calculate the safety stock by SKU, where the inventory type is a finished good (FG) and the stocking type is made-to-stock (MTS).

It is also important to notate the formula for calculating safety stock, which is below:

$$ SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

Knowing this, we know that the key features of this dataset would include the SKU, inventory type, stocking type, demand/quantity, and lead time.

## Question 1: DataFrame Formatting

As you load and inspect your transaction dataframe, you'll notice the column names don't follow a consistent convention. Make the necessary changes to ensure all column names follow the same convention as the metadata.

### Answer 1:

#### Headers for meta_data
We want to first get the headers for the meta data in order to identify what needs to be corrected or matched. We also want to get a description of the feature/header for our actual transactional dataset.

We want to display the metadata table first.

In [138]:
meta_data

,feature,description
0,sku_number,Eight character stock keeping unit number
1,inventory_type,Types of inventory including finished good - w...
2,stocking_type,Stockign type policy including make to stock -...
3,lead_time,Total lead time in days
4,unit_price,Unit price US dollar value
5,manufacturing_site,Manufacturing site ID starting with country co...
6,division_code,Four character business division ID
7,transaction_date,Date of Transaction in Year-Month-Day format
8,order_quantity,Order quantity - rounded up for convenience


#### Meta_data Analysis

After looking at the table, we see that there are 9 features with the following data types:

| Feature | Data Type | 
| -------- | ---------- |
| sku_number | Qualitative Nominal |
| inventory_type | Qualitative Ordinal |
| stocking_type | Qualitative Nominal |
| lead_time | Quantitative Discrete |
| unit_price | Quantitative Continuous |
| manufacturing_site | Qualitative Nominal |
| division_code | Qualitative Nominal |
| transaction_date | Qualitative Ordinal |
| order_quantity | Quantitative Discrete |

Now, we'll check the transaction_data to see if the headers and the data types match.


In [139]:
transaction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 711662 entries, 0 to 711661
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          711249 non-null  str    
 1   inventory_type      698248 non-null  str    
 2   stocking_type       697858 non-null  str    
 3   leadTime            711662 non-null  int64  
 4   unit_price          711662 non-null  float64
 5   manufacturing_site  573584 non-null  str    
 6   division Code       661808 non-null  str    
 7   transaction_date    711662 non-null  str    
 8   Order-Quantity      711662 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 48.9 MB


#### Cross-Check & Creating Dictionary

Now we see that generally, all of the data type matches. The only one which we may need to perform some transformation on is "transaction_date".
As far as the headers, it looks like "lead_time", "division_code", and "order_quantity" need to be updated. But just in case, let's go ahead and rename the columns by overwriting the existing column names.

In [140]:
transaction_title = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]

In [141]:
transaction_data.columns = transaction_title

#### Validation for new column names

Next, we will look at the transaction data info to ensure that the correct headers were used.

In [142]:
transaction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 711662 entries, 0 to 711661
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          711249 non-null  str    
 1   inventory_type      698248 non-null  str    
 2   stocking_type       697858 non-null  str    
 3   lead_time           711662 non-null  int64  
 4   unit_price          711662 non-null  float64
 5   manufacturing_site  573584 non-null  str    
 6   division_code       661808 non-null  str    
 7   transaction_date    711662 non-null  str    
 8   order_quantity      711662 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 48.9 MB


## Question 2: 
Inspect the numeric features of your dataframe with the describe function. Provide a summary of these features, and in case there are any oddities/anomalies in your dataframe, locate and correct them.  Please include a note on any oddities you find and a rationale for your suggested correction. You can do so by including a text note in a corresponding markdown shell.

In [143]:
transaction_data.describe()

,lead_time,unit_price,order_quantity
count,711662.000000,711662.000000,711662.000000
mean,25.771684,797.880175,79.788738
std,10.704492,375.268727,41.964285
min,14.000000,-1015.534324,-121.000000
25%,28.000000,982.030704,63.000000
50%,28.000000,996.396930,93.000000
75%,28.000000,1004.287522,109.000000
max,2800.000000,1029.941378,198.000000


### Answer 2:

We see that after running the describe function, there are negative numbers in the "unit_price" and "order_quantity". 
We also see a maximum lead time of 2800. We would want to also investigate any lead times which are beyond 3 standard deviations away from the mean.

#### Order Quantity less than 0

We see that we have 36,437 observations that have a negative order quantity. Looking at the inventory type, we see a mix of finished goods (FG), work-in-progress (WIP), and raw materials (RM). 

Typically, these types of transactions would indicate raw materials which were returned to the supplier, items in WIP which perhaps had a production order (but no manufacturing actually started) that was cancelled, or actual finished goods which were returned by the customer, or even identify a quality issue where items in WIP or even FGs had to be returned/canceled. It could also indicate excess materials which were scrapped.

Normally, we would keep these transactions as is, as it is common for materials to continuously move back and forth between the production lines. Since the ultimate goal of this entire exercise it to calculate safety stock, we need to look at the columns/variables that are used in the calculation of safety stock. This would include demand (quantity) and lead time. 

We performed further analysis below (describing the data for quantities less than 0, and for those greater than 0), and compared these against the entire dataset. We concluded that the mean, standard deviation, and lead time would not change by much.

Even if these transactions were returns or cancellations, the organization still manufactured the product and its replacements if a replacement was needed. Thus, we would consider **dropping** these values.

In [144]:
transaction_data.query('order_quantity < 0')

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,0CD5B7B2,WIP,MTO,14,42.905172,NaN,CEA3,2023-01-01,-9
4,9D6AC92B,RM,MTS,14,46.998541,NaN,1C8A,2023-01-01,-8
10,D701F1C7,RM,MTO,14,44.999667,CA-14,1C8A,2023-01-01,-16
63,1990512A,RM,MTS,14,51.537381,PL-38,1536,2023-01-01,-17
69,DFFADF85,FG,MTO,14,52.500816,PL-8F,NaN,2023-01-01,-15
...,...,...,...,...,...,...,...,...,...
711604,B3A50BFB,WIP,MTO,14,51.076793,CA-B4,B395,2025-12-30,-10
711610,99E3E090,FG,MTS,14,44.573580,NaN,BE87,2025-12-30,-15
711612,CEA0C29C,RM,ATO,21,200.918635,NaN,D989,2025-12-30,-5
711626,6B0E4EEF,FG,MTO,14,48.578219,PL-8F,1536,2025-12-30,-12


In [145]:
transaction_data.describe()

,lead_time,unit_price,order_quantity
count,711662.000000,711662.000000,711662.000000
mean,25.771684,797.880175,79.788738
std,10.704492,375.268727,41.964285
min,14.000000,-1015.534324,-121.000000
25%,28.000000,982.030704,63.000000
50%,28.000000,996.396930,93.000000
75%,28.000000,1004.287522,109.000000
max,2800.000000,1029.941378,198.000000


In [146]:
transaction_data.query('order_quantity >= 0').describe()

,lead_time,unit_price,order_quantity
count,675225.000000,675225.000000,675225.000000
mean,26.265708,836.183670,84.508756
std,10.723942,345.760154,37.664268
min,14.000000,-1015.534324,0.000000
25%,28.000000,986.112631,72.000000
50%,28.000000,997.110922,94.000000
75%,28.000000,1004.808758,110.000000
max,2800.000000,1029.941378,198.000000


In [147]:
transaction_data_clean = transaction_data.query('order_quantity >= 0')

#### Order Quantity is 0
We also want to make sure that our data does not contain any order quantities where it is 0. Including these can often skew both the order quantity mean, and the lead time. If no actual order quantity was placed, we would consider this as an data entry error, or it could also indicate a cancellation. Rather than removing the transaction, it could be well-intended for the company's policy to still keep the transaction record, but update the quantity to 0. Nonetheless, for the purposes of our goal, it should be **omitted**.

In [148]:
transaction_data_clean.query('order_quantity == 0').describe()

,lead_time,unit_price,order_quantity
count,4148.000000,4148.000000,4148.0
mean,16.738910,80.791172,0.0
std,4.753034,55.894166,0.0
min,14.000000,32.909604,0.0
25%,14.000000,48.131864,0.0
50%,14.000000,52.730241,0.0
75%,21.000000,97.850171,0.0
max,28.000000,225.605428,0.0


In [149]:
transaction_data_clean = transaction_data_clean.query('order_quantity != 0')
transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,671077.000000,671077.000000,671077.000000
mean,26.324595,840.852833,85.031114
std,10.724257,341.644368,37.188021
min,14.000000,-1015.534324,1.000000
25%,28.000000,986.262413,73.000000
50%,28.000000,997.271402,94.000000
75%,28.000000,1004.881084,110.000000
max,2800.000000,1029.941378,198.000000


#### Unit Price less than 0

For the unit pricing, there could be a plethora of reasons as to why we see a negative unit price. They could be cost adjustments, supplier rebates, or it could also be a typo. Cost adjustments could be related to supplier-related cost adjustments, overhead/labor adjustments, or depending on the ERP system and the type of return, the transaction could appear as a negative unit price with a positive order quantity. Some organizations may want to see supplier rebates or cost adjustments differently than actual returns, to help with differentiating what type of "negative" inventory balance occurred. For example, at Amkor Technology, we tend to keep some transaction line items with a positive quantity but a negative unit price to indicate a credit that was issued to our customer for reconciliation purposes. Cost adjustments of raw materials are also transacted in a similar manner, where the quantity is left intact, but the unit price reflects a negative value to issue a credit to the supplier.

If more context is provided and these are actually errors, we would request corrections on these, or just take the mean or median of the unit price by SKU and inventory type.

However, we run into the same issues as the negative quantities. If the ultimate goal is to calculate the safety stock, any credits or cost adjustments should not be included in the order quantity. If these are truly rebates, reconciliations, or adjustments, we don't want the "duplicate" lead times to be included in our safety stock calculation. **In the grand scheme of things, we only have 10 transactions with a negative unit price. Thus, our suggestion is to REMOVE these transaction rows, as the order quantity and lead time values would not "move the needle" much.**

In [150]:
transaction_data_clean.query('unit_price < 0').describe()

,lead_time,unit_price,order_quantity
count,10.000000,10.000000,10.000000
mean,27.300000,-920.553034,98.600000
std,2.213594,253.867643,25.695655
min,21.000000,-1015.534324,34.000000
25%,28.000000,-1004.778647,92.000000
50%,28.000000,-999.283847,107.000000
75%,28.000000,-995.216109,113.500000
max,28.000000,-198.417515,123.000000


##### Removing Negative Unit Price

In [151]:
transaction_data_clean = transaction_data_clean.query('unit_price >= 0')
transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,671067.000000,671067.000000,671067.000000
mean,26.324580,840.879081,85.030912
std,10.724334,341.577978,37.188142
min,14.000000,32.909604,1.000000
25%,28.000000,986.262413,73.000000
50%,28.000000,997.271402,94.000000
75%,28.000000,1004.881084,110.000000
max,2800.000000,1029.941378,198.000000


#### Unit price = 0

Next, we also want to look at anything with a unit price of 0. There is **no data** with a unit price of 0. No further action is needed.

In [152]:
transaction_data_clean.query('unit_price == 0').describe()

,lead_time,unit_price,order_quantity
count,0.0,0.0,0.0
mean,NaN,NaN,NaN
std,NaN,NaN,NaN
min,NaN,NaN,NaN
25%,NaN,NaN,NaN
50%,NaN,NaN,NaN
75%,NaN,NaN,NaN
max,NaN,NaN,NaN


#### Lead time greater than 3 standard deviations from the mean (outliers)

While the describe function from pandas already gave us the values for the mean and standard deviation for lead time, I don't want to manually write down and calculate the number to filter. Thus, I've also loaded numpy to perform the calculation, set 2 variables called "lt_3std" and "lt_mean" and using those variables to apply my filter.

In [153]:
lt_3std = 3*np.std(transaction_data_clean["lead_time"])
lt_mean = np.mean(transaction_data_clean["lead_time"])

In [154]:
transaction_data_clean.query('abs(lead_time - @lt_mean) > @lt_3std').describe()

,lead_time,unit_price,order_quantity
count,10.00000,10.000000,10.000000
mean,2520.00000,814.230902,84.900000
std,590.29183,402.870461,39.478405
min,1400.00000,47.259863,13.000000
25%,2800.00000,995.972464,72.750000
50%,2800.00000,1000.637763,96.000000
75%,2800.00000,1008.420766,114.750000
max,2800.00000,1017.014895,123.000000


##### Checking for any lead times with a "0" value

In [155]:
transaction_data_clean.query('lead_time == 0').describe()

,lead_time,unit_price,order_quantity
count,0.0,0.0,0.0
mean,NaN,NaN,NaN
std,NaN,NaN,NaN
min,NaN,NaN,NaN
25%,NaN,NaN,NaN
50%,NaN,NaN,NaN
75%,NaN,NaN,NaN
max,NaN,NaN,NaN


##### Lead Time Corrections

Based on the lead times that were in the transactional data, it is **extremely** unlikely that any lead time would be 3.5 to 7.7 years. Our intuition would tell us to *omit* or remove these line items, or to replace the lead time with the mean or median of the lead time of the sku_number and inventory type (omitting the outliers).

Considering that we only have 10 transactions where the lead time is greater than 1400 days, we would choose to remove these from our dataset.

*Note: The reason why we would not just look at the sku_number is because one SKU can have several inventory types, depending on how they are classified in each manufacturing process. As we add more "value" or "time" with each process, the value may change*

In [156]:
transaction_data_clean = transaction_data_clean.query('abs(lead_time - @lt_mean) <= @lt_3std')
transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,671057.000000,671057.000000,671057.000000
mean,26.287420,840.879478,85.030914
std,4.203842,341.577321,37.188138
min,14.000000,32.909604,1.000000
25%,28.000000,986.262413,73.000000
50%,28.000000,997.271402,94.000000
75%,28.000000,1004.881084,110.000000
max,28.000000,1029.941378,198.000000


## Question 3 — NaN Values
Your transactional dataset includes many missing values. Find and address (in your dataframe) the missing values by column. Again, remember to note the NaN values you found and provide a rationale for your suggested action.

### Answer 3:
We want to go through every column and sum the total numbers of NaN values in each column. Since we also REMOVED some rows from the previous step(s), we would also re-reun the isna().sum() function after removing to get the "corrected" values for the counts.

#### SKU Number
Since the SKU number is generally a **unique** identifier, we would NOT fill in the data. However, since we're only interested in calculating the safety stock by SKU and stocking type, we would ultimately opt to **remove** these rows.

However, it should be noted that it ultimately does not matter if we choose to keep these transaction rows as is, or replacing with a value of "Unknown". When we perform a calculation of the safety stock by SKU and inventory type, we would just have a safety stock calculation, but it does not carry any real meaning.

In [157]:
transaction_data_clean.isna().sum()

sku_number               298
inventory_type         12511
stocking_type          13224
lead_time                  0
unit_price                 0
manufacturing_site    130239
division_code          47037
transaction_date           0
order_quantity             0
dtype: int64

##### Removing NaN values from SKU

In [158]:
transaction_data_clean = transaction_data_clean[transaction_data_clean["sku_number"].notna()]
transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,670759.000000,670759.000000,670759.000000
mean,26.289873,841.220363,85.063686
std,4.200576,341.269604,37.163560
min,14.000000,32.909604,1.000000
25%,28.000000,986.262413,73.000000
50%,28.000000,997.271402,94.000000
75%,28.000000,1004.881084,110.000000
max,28.000000,1029.941378,198.000000


#### Inventory Type
Inventory type has 12,511 transactions with missing values. 

However, since the ultimate goal of this exercise is to calculate safety stock by SKU for FG MTS products, there is no way for us to accurately "fill" this information in. We don't want to remove more items than necessary. We will later filter for any transactions with and inventory type of "FG", so we would ultimately leave the transactions alone.

If there is a future request to also calculate the safety stock based on inventory type, we would perform further clean-up measures to fill in the information, or omit the transactions from our dataset.

In [159]:
transaction_data_clean.isna().sum()

sku_number                 0
inventory_type         12511
stocking_type          13224
lead_time                  0
unit_price                 0
manufacturing_site    130188
division_code          47011
transaction_date           0
order_quantity             0
dtype: int64

#### Stocking Type
While stocking type is critical for us to filter and calculate safety stock by FG and MTS stocking types, we cannot accurately "fill" in the NaN values for stocking type, since each SKU can carry mutliple stocking types. Thus, we would opt to to **keep** the dataset as it, with a NaN value. We could also remove these rows, but the less that we touch the original dataset, the better.

If there was another request in the future to further cleanse the data, we would either opt to fill or remove these transaction rows.

In [160]:
transaction_data_clean.isna().sum()

sku_number                 0
inventory_type         12511
stocking_type          13224
lead_time                  0
unit_price                 0
manufacturing_site    130188
division_code          47011
transaction_date           0
order_quantity             0
dtype: int64

#### Lead Time
No NaN values for leadtime. No action needed.

#### Unit Price
No NaN values for unit price. No action needed.

#### Manufacturing Site
Again, this feature has no bearing in terms of calculating the safety stock by SKU and stocking type. Since the NaN values does contain important transactions that does carry weight/bearing on our safety stock calculation goal, we would choose to **keep** the transaction data, but **not** fill in the data.

In [161]:
transaction_data_clean["manufacturing_site"].isna().sum()

np.int64(130188)

In [162]:
transaction_data_clean[transaction_data_clean["manufacturing_site"].isna()]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
7,84FCC369,FG,MTS,28,977.181212,NaN,D9AC,2023-01-01,90
11,A99A8E8D,FG,MTO,28,994.253207,NaN,1C8A,2023-01-01,93
16,5AC3C06B,FG,MTO,28,1015.070220,NaN,1536,2023-01-01,98
27,989A2064,FG,MTS,21,189.249174,NaN,A786,2023-01-01,38
28,B1906A71,RM,MTO,28,996.940154,NaN,BE87,2023-01-01,105
...,...,...,...,...,...,...,...,...,...
711639,261DBD1E,FG,ATO,28,1009.218861,NaN,CEA3,2025-12-30,76
711646,492AD923,WIP,NaN,28,1001.018361,NaN,CC8E,2025-12-30,110
711652,CEA0C29C,FG,MTO,28,1006.562612,NaN,1429,2025-12-30,67
711654,C8EADE59,FG,MTS,28,1000.570170,NaN,F049,2025-12-30,94


#### Division Code

Again, this feature has no bearing in terms of calculating the safety stock by SKU and stocking type. Since the NaN values does contain important transactions that does carry weight/bearing on our safety stock calculation goal, we would choose to **keep** the transaction data, but **not** fill in the data.

In [163]:
transaction_data_clean["division_code"].isna().sum()

np.int64(47011)

In [164]:
transaction_data_clean[transaction_data_clean["division_code"].isna()]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
3,9EC6C209,WIP,MTS,28,1009.830241,CA-B4,NaN,2023-01-01,122
6,E4AC6BAC,FG,MTO,28,1002.601466,CA-4C,NaN,2023-01-01,134
40,7D780403,RM,MTS,28,999.779816,PL-B2,NaN,2023-01-01,117
46,A5A40F71,WIP,MTS,28,997.426100,PL-8F,NaN,2023-01-01,117
68,DD969A17,FG,MTS,14,52.179130,PL-B2,NaN,2023-01-01,13
...,...,...,...,...,...,...,...,...,...
711547,CEA0C29C,FG,MTO,28,1006.562612,CA-B4,NaN,2025-12-30,112
711558,A3095E6C,WIP,MTO,28,989.006415,PL-8F,NaN,2025-12-30,107
711613,9A73ED78,FG,NaN,28,993.774518,CA-4C,NaN,2025-12-30,99
711628,0D4A17C7,WIP,MTO,28,1006.687408,CA-B4,NaN,2025-12-30,124


#### Transaction Date
No NaN values for transaction date. No action needed.

In [165]:
transaction_data_clean["transaction_date"].isna().sum()

np.int64(0)

#### Order Quantity
No NaN values for order quantity. No action needed.

In [166]:
transaction_data_clean["order_quantity"].isna().sum()

np.int64(0)

#### Summary of NaN counts for each feature after filtering from Question 2

| Feature | Count of NaN | 
| -------- | ---------- |
| sku_number | 298 |
| inventory_type | 12,511 |
| stocking_type | 13,224 |
| lead_time | 0 |
| unit_price | 0 |
| manufacturing_site | 130,188 |
| division_code | 47,011 |
| transaction_date | 0 |
| order_quantity | 0 |

#### Summary of Decisions for each feature
For SKU number, since it is a unique identifier, we would not fill in the data. Considering the context of our goal to calculate the safety stock by SKU and stocking type, it ultimately does not matter if we keep or remove the rows. We would choose to just keep it for now.

For stocking type and inventory type, these features are needed in order to appropriately filter the data. However, there is no accurate way to "fill" the data in. Thus, we would just leave the rows alone, but we would **not** remove the transactions from the dataset. Our position is that the less that we "touch" the original data, the better.

For manufacturing site and divison code, we would keep the data as is, since this feature is not needed for calculating safety stock by SKU and stocking type. However, we would **NOT** remove the transactions from the dataset, since it does contain rows that have key information for our ultimate goal.

Lead time, unit price, transaction date, and order quantity would require no transformation, filling, or removing of NaN values, since there are **no** NaN values.


## Question 4 — Extracting The Country Values

To apply broader breakdowns of your transactional dataset, it will be helpful to add a country column. Recall that the metadata indicates the country information is somewhat captured in the site column. Create a new country column by using the manufacturing_site information. To do so, you will need the helper function str.startswith() (here is a [link](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.startswith.html) to the official documentation), which is defined for a pandas Series object. As the documentation states, str.startswith() tests if the start of each string element matches a given pattern.

### Answer 4:
From the metadata, we know that the countries that we are interested in are USA, Canada, Poland, and Japan.

#### Find unique values under manufacturing site
Next, we want to create a list of the manufacturing sites. We see that there is a pattern, where PL = Poland, US = USA, CA = Canada, and JP = Japan. Rather than mapping out each unique value, we use this pattern to see if we can use the "start with" function. Ideally, we'd like to just see the values of the first 2 characters of the string to then map it to a dictionary.

In [167]:
transaction_data_clean["manufacturing_site"].unique()

<StringArray>
['PL-38', 'PL-B2', 'CA-B4', 'CA-4C',     nan, 'US-6B', 'PL-8F', 'JP-84',
 'PL-0A', 'PL-A2', 'CA-14', 'JP-96', 'JP-38', 'PL-2D', 'US-2B', 'US-3A',
 'PL-F8', 'JP-F5', 'PL-96', 'JP-67']
Length: 20, dtype: str

#### Creating a dictionary
In order to add a column based on the values of the manufacturing site, we need to create a dictionary first.

In [168]:
country_map = {"CA" : "Canada",
               "JP" : "Japan",
               "PL" : "Poland",
               "US" : "USA"}

#### Mapping to new country
Creates a new column named "country", we take it from the manufacturing site, look at the string of the first 2 characters,and map it to our dictionary. We learned in class that the str function converts the selected column values into a string expression, and creating a index from ":2" would allow the system to select the manufacturing site, transform it into a string expression, and to grab the first 2 characters of the string. Since Python starts a 0, the index would stop **before** the 2nd character (character 0 and character 1). *(Stack Overflow, 2022)*

In [169]:
transaction_data_clean["country"] = transaction_data_clean["manufacturing_site"].str[:2].map(country_map)

In [170]:
transaction_data_clean.sample(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
161806,9450F1ED,FG,MTO,28,986.849065,PL-8F,BE87,2023-09-06,82,Poland
172378,A12C546B,RM,MTO,28,1000.577042,PL-B2,A786,2023-09-22,59,Poland
446651,21A7B08C,WIP,MTS,28,1000.702379,NaN,A786,2024-11-17,115,NaN
36757,CC3943E9,FG,MTS,28,1005.050849,NaN,929A,2023-02-26,67,NaN
530008,5978F66E,FG,MTO,28,984.432806,PL-B2,1536,2025-03-25,91,Poland
168915,64906A31,RM,MTO,28,1001.168338,JP-F5,A786,2023-09-16,99,Japan
313640,075C888F,FG,MTS,28,1008.202235,PL-B2,F049,2024-04-27,99,Poland
75325,9450F1ED,FG,MTO,28,986.849065,NaN,CEA3,2023-04-26,99,NaN
441330,D4B45355,FG,MTO,28,990.026524,NaN,9E13,2024-11-09,95,NaN
90965,C8B1B06C,FG,MTS,14,50.240397,NaN,CEA3,2023-05-20,6,NaN


## Question 5 — Helpful Dimensional Information
It's time to focus on the subset of this dataset we need. Since we ultimately want to calculate safety stock, focus on finished goods designated as "make to stock." Create a filtered subset of your transaction dataset and write appropriate code to answer the following questions (for the filtered data).

+ How many unique SKUs are we working with?
+ How many unique Manufacturing Sites are we working with?
+ How many Divisions are we working with?
+ What are the top and bottom 10 transactions by order quantity?
+ What are the top and bottom 10 transactions by total sales value? Hint: Sales is not a feature in this dataset, so we must calculate it using the existing features.

### Answer 5:
#### Creating a filtered subset
We want to create a new subset called "MTS_data" where we are working with the transactions where the stocking_type is only "MTS" and the inventory_type is "FG".

In [171]:
MTS_data = transaction_data_clean.query('stocking_type == "MTS" and inventory_type == "FG"')
MTS_data

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
1,FED6CA5E,FG,MTS,28,1019.701380,PL-38,1536,2023-01-01,63,Poland
7,84FCC369,FG,MTS,28,977.181212,NaN,D9AC,2023-01-01,90,NaN
8,7E0B9A17,FG,MTS,28,999.436684,US-6B,BE87,2023-01-01,99,USA
12,8A1F415E,FG,MTS,28,1005.366262,CA-B4,1C8A,2023-01-01,78,Canada
13,90D5E95A,FG,MTS,14,41.499619,PL-8F,1536,2023-01-01,7,Poland
...,...,...,...,...,...,...,...,...,...,...
711636,F916CAAF,FG,MTS,28,1019.992250,JP-67,F049,2025-12-30,95,Japan
711638,7E0B9A17,FG,MTS,28,991.088830,PL-B2,CEA3,2025-12-30,100,Poland
711643,00874431,FG,MTS,28,990.146733,PL-B2,D9AC,2025-12-30,125,Poland
711654,C8EADE59,FG,MTS,28,1000.570170,NaN,F049,2025-12-30,94,NaN


#### Unique SKUs Count
To see the unique number of SKUs in our filtered subset, we want to do a value_counts. This value_counts will also automatically *ignore* NaN entries. We also want to find the LENGTH, so we use the len function.

We have 375 unique SKUs. Since we already "dropped" the NaN values, we only have 375 unique SKUs.

In [172]:
len(MTS_data["sku_number"].value_counts())

375

#### Unique manufacturing sites
We perform the same exercise with the SKUs. There are 19 unique manufacturing sites, 20 if including NaN.

In [173]:
len(MTS_data["manufacturing_site"].value_counts())

19

In [174]:
len(MTS_data["manufacturing_site"].value_counts(dropna=False))

20

#### Unique divisions

There are 54 unique divisions for MTS transactions. 55 if including NaN.

In [175]:
len(MTS_data["division_code"].value_counts())

54

In [176]:
len(MTS_data["division_code"].value_counts(dropna=False))

55

#### Finding top 10 transactions by order quantity

In [177]:
MTS_data.sort_values("order_quantity", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
33499,7E0B9A17,FG,MTS,28,991.088830,CA-B4,CEA3,2023-02-21,198,Canada
185631,FDDDA064,FG,MTS,28,1010.068090,JP-38,C04F,2023-10-12,194,Japan
180871,285D1D8D,FG,MTS,28,1000.289315,CA-B4,D9AC,2023-10-05,186,Canada
406049,E045018B,FG,MTS,28,995.605558,PL-0A,D9AC,2024-09-16,184,Poland
393301,578DD826,FG,MTS,28,1019.132513,CA-B4,DC0A,2024-08-27,181,Canada
155365,B876EA18,FG,MTS,28,1002.119781,NaN,B395,2023-08-27,180,NaN
278589,5C704F7E,FG,MTS,28,987.620742,PL-8F,1536,2024-03-04,179,Poland
671624,ACC4CB10,FG,MTS,28,998.841700,PL-B2,CB7E,2025-10-30,178,Poland
642523,7D1D8544,FG,MTS,28,996.942519,JP-38,A786,2025-09-15,178,Japan
641161,5E589032,FG,MTS,28,1006.262150,NaN,CEA3,2025-09-13,178,NaN


#### Finding bottom 10 transactions by order quantity
Since we also may want to see any non-negative transactions to exclude the credits or returns, we may want to opt to exclude looking at those.

In [178]:
MTS_data.query('order_quantity > 0').sort_values("order_quantity").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
343658,FDDDA064,FG,MTS,14,54.277874,PL-8F,57A9,2024-06-12,1,Poland
405082,B798C3DD,FG,MTS,21,173.792807,NaN,1536,2024-09-14,1,NaN
417468,F9B8CB39,FG,MTS,14,51.835739,NaN,F049,2024-10-03,1,NaN
417854,5C78C11F,FG,MTS,14,56.861257,CA-4C,D9AC,2024-10-04,1,Canada
333,2FA073AE,FG,MTS,14,49.391035,CA-B4,C04F,2023-01-01,1,Canada
325468,5E589032,FG,MTS,21,208.846161,CA-4C,BE87,2024-05-15,1,Canada
247267,91D8FD9E,FG,MTS,14,47.992674,NaN,CB7E,2024-01-15,1,NaN
319834,AC3ADABF,FG,MTS,21,200.715983,PL-A2,A80E,2024-05-07,1,Poland
465236,F2A4F89E,FG,MTS,14,58.020544,PL-B2,F049,2024-12-15,1,Poland
364876,E21018B4,FG,MTS,28,100.030294,PL-38,D9AC,2024-07-15,1,Poland


#### Adding total sales value column
We add a new column titled "total_sales_value" and multiply the order quantity by the unit price.

In [179]:
MTS_data["total_sales_value"] = MTS_data["order_quantity"]*MTS_data["unit_price"]

In [180]:
MTS_data

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,total_sales_value
1,FED6CA5E,FG,MTS,28,1019.701380,PL-38,1536,2023-01-01,63,Poland,64241.186944
7,84FCC369,FG,MTS,28,977.181212,NaN,D9AC,2023-01-01,90,NaN,87946.309077
8,7E0B9A17,FG,MTS,28,999.436684,US-6B,BE87,2023-01-01,99,USA,98944.231693
12,8A1F415E,FG,MTS,28,1005.366262,CA-B4,1C8A,2023-01-01,78,Canada,78418.568415
13,90D5E95A,FG,MTS,14,41.499619,PL-8F,1536,2023-01-01,7,Poland,290.497336
...,...,...,...,...,...,...,...,...,...,...,...
711636,F916CAAF,FG,MTS,28,1019.992250,JP-67,F049,2025-12-30,95,Japan,96899.263785
711638,7E0B9A17,FG,MTS,28,991.088830,PL-B2,CEA3,2025-12-30,100,Poland,99108.882956
711643,00874431,FG,MTS,28,990.146733,PL-B2,D9AC,2025-12-30,125,Poland,123768.341576
711654,C8EADE59,FG,MTS,28,1000.570170,NaN,F049,2025-12-30,94,NaN,94053.595963


#### Top 10 transactions by total sales value
Now that we've added a new column, we do the same sorting functions as above.

In [181]:
MTS_data.sort_values("total_sales_value", ascending = False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,total_sales_value
33499,7E0B9A17,FG,MTS,28,991.088830,CA-B4,CEA3,2023-02-21,198,Canada,196235.588252
185631,FDDDA064,FG,MTS,28,1010.068090,JP-38,C04F,2023-10-12,194,Japan,195953.209533
180871,285D1D8D,FG,MTS,28,1000.289315,CA-B4,D9AC,2023-10-05,186,Canada,186053.812682
393301,578DD826,FG,MTS,28,1019.132513,CA-B4,DC0A,2024-08-27,181,Canada,184462.984874
406049,E045018B,FG,MTS,28,995.605558,PL-0A,D9AC,2024-09-16,184,Poland,183191.422709
155365,B876EA18,FG,MTS,28,1002.119781,NaN,B395,2023-08-27,180,NaN,180381.560506
669165,8ED02F71,FG,MTS,28,1013.010673,JP-67,NaN,2025-10-26,177,Japan,179302.889117
641161,5E589032,FG,MTS,28,1006.262150,NaN,CEA3,2025-09-13,178,NaN,179114.662782
253147,6B527587,FG,MTS,28,1011.589319,PL-38,BE87,2024-01-24,176,Poland,178039.720095
671624,ACC4CB10,FG,MTS,28,998.841700,PL-B2,CB7E,2025-10-30,178,Poland,177793.822599


#### Bottom 10 transactions by total sales value

Although our team made the decision to include negative quantities, we thought it might be best to filter out any negative values, as typically we just want to see what is the lowest sales value. Since these possible indicate credits or returns, we also provided another table with the lowest sales values that are non-negative. Note that we already removed the transactions with a quantity of 0.

In [182]:
MTS_data.sort_values("total_sales_value").head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,total_sales_value
335738,B76BF308,FG,MTS,14,34.391014,PL-8F,1C8A,2024-05-31,1,Poland,34.391014
183845,B76BF308,FG,MTS,14,34.391014,PL-B2,929A,2023-10-09,1,Poland,34.391014
261559,B76BF308,FG,MTS,14,34.391014,NaN,D9AC,2024-02-06,1,NaN,34.391014
285368,63C4377B,FG,MTS,14,37.043908,PL-8F,C04F,2024-03-15,1,Poland,37.043908
551023,63C4377B,FG,MTS,14,37.043908,PL-B2,1536,2025-04-26,1,Poland,37.043908
98491,56D83D7A,FG,MTS,14,37.669826,NaN,CEA3,2023-06-01,1,NaN,37.669826
638470,56D83D7A,FG,MTS,14,37.669826,PL-B2,C04F,2025-09-09,1,Poland,37.669826
420016,369A4789,FG,MTS,14,38.382045,NaN,BE87,2024-10-07,1,NaN,38.382045
21452,369A4789,FG,MTS,14,38.382045,PL-8F,CDD0,2023-02-02,1,Poland,38.382045
376142,369A4789,FG,MTS,14,38.382045,CA-4C,CB7E,2024-08-01,1,Canada,38.382045


## References

Stack Overflow. (2022, March 16). How to substring values from a pandas df [Online forum post]. Stack Exchange Inc. https://stackoverflow.com/questions/71537795/how-to-substring-values-from-a-pandas-df